# 1c′ - Hư hại ngữ pháp có GÂY RA sụt F1 không?

Trên CPU với reader **Qwen2.5-0.5B**, tương quan giữa tỉ số hư-từ và F1 là
**ρ = +0.03**, CI [−0.11, +0.18] - không có tương quan. Nhưng 59–66% câu có
F1 = 0, nên **hiệu ứng sàn** có thể đang nuốt tín hiệu: không phân biệt được
"hỏng vì mất ngữ pháp" với "hỏng vì reader quá nhỏ".

Notebook này chạy lại với **Qwen2.5-7B 4-bit** (F1 trung bình ~43 thay vì ~25).
Nếu tương quan xuất hiện, hư hại ngữ pháp thành **nguyên nhân** chứ không chỉ là
thuộc tính đo được - đó là điều quyết định mục 2.3 của proposal viết mạnh hay yếu.

**Kết quả có thể vẫn âm.** Trường hợp đó cũng là kết quả: nó nói hư hại ngữ pháp
không giải thích được chênh lệch F1, và ta phải tìm cơ chế khác.

Trước khi chạy: **Settings → Accelerator → GPU T4 x2**, và **Internet → On**.

In [ ]:
!pip install -q llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -2


In [ ]:
# Lấy mã nguồn từ gist. Repo là private nên Colab không clone ẩn danh được;
# gist thì public, tải thẳng, không phải upload tay lần nào. URL không kèm SHA
# nên luôn trỏ bản mới nhất.
import os, sys, base64, io, zipfile, urllib.request, shutil

# Thư mục làm việc: Colab dùng /content, Kaggle dùng /kaggle/working. Phát hiện
# thay vì cứng hoá - cùng notebook chạy được cả hai, và trên máy khác thì lùi về
# thư mục hiện tại.
BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'repo')
SRC_URL = ('https://gist.githubusercontent.com/nhantrnh/'
           'aeec3610fb9541b5f8b388cafd424eb9/raw/itercomp_src_b64.txt')
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data',
        'metrics', 'fertility', 'stats', 'budget']

def missing_in(root):
    return [m for m in NEED
            if not os.path.exists(os.path.join(root, 'src', 'itercomp', m + '.py'))]

# Ra khỏi REPO trước khi xoá: nếu lần chạy trước đã os.chdir(REPO) thì cwd đang
# nằm trong thư mục sắp bị xoá, và mọi thao tác mở file sau đó sẽ ném
# FileNotFoundError khó hiểu.
os.chdir(BASE)

# Xoá sạch: extractall KHÔNG tự xoá file cũ, nên một gói cũ có thể "sống sót"
# qua nhiều lần chạy và che mất bản mới.
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]          # bỏ module đã nạp trong bộ nhớ
os.makedirs(REPO, exist_ok=True)

with urllib.request.urlopen(SRC_URL, timeout=60) as r:
    blob = base64.b64decode(r.read())
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    z.extractall(REPO)
print(f'✓ tải mã nguồn từ gist ({len(blob)/1024:.0f} KB)')

miss = missing_in(REPO)
if miss:
    raise SystemExit(f"Gói mã nguồn thiếu module: {', '.join(miss)}")

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

from itercomp import itercomp, load_dataset, normalize_boolean, budget_filter
print('cwd:', os.getcwd(), '| scripts/:', os.path.isdir('scripts'))
print(f'✓ đủ {len(NEED)} module')

In [ ]:
import urllib.request
FILES = {
 'vimqa/validation.parquet':    ('nguyenlab/vimqa', 'data/validation-00000-of-00001.parquet'),
 '2wiki/dev.parquet':           ('xanhho/2WikiMultihopQA', 'dev.parquet'),
 'hotpotqa/validation.parquet': ('hotpotqa/hotpot_qa', 'distractor/validation-00000-of-00001.parquet'),
 'musique/dev.jsonl':           ('dgslibisey/MuSiQue', 'musique_ans_v1.0_dev.jsonl'),
}
for dst, (repo, src) in FILES.items():
    p = 'data/' + dst
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        urllib.request.urlretrieve(f'https://huggingface.co/datasets/{repo}/resolve/main/{src}', p)
    print(f'{dst:32s} {os.path.getsize(p)/1e6:6.1f} MB')

## 1. Chạy bảng chính n=200 với reader 7B

Cần `per_row` cho cả `itercomp` lẫn `llmlingua2` - đó là đầu vào của bước 2.

In [ ]:
# ══ LẤY vimqa_200_7b.json ══
# Kaggle KHÔNG chia sẻ output giữa các kernel qua đường dẫn. Nếu file không có,
# phải nói rõ và DỪNG - đừng âm thầm chạy lại bảng chính.
#
# Đã xảy ra thật: một kernel CPU không tìm thấy file, rơi vào nhánh chạy lại,
# rồi bị Kaggle hủy sau hơn một giờ vì reader 7B cần GPU. Mất một phiên vì một
# nhánh dự phòng tưởng là hữu ích.
import os, sys, glob, shutil

os.makedirs('results', exist_ok=True)
OUT = 'results/vimqa_200_7b.json'

if not os.path.exists(OUT):
    hits = glob.glob('/kaggle/input/**/vimqa_200_7b.json', recursive=True)
    if hits:
        shutil.copy(hits[0], OUT)
        print(f'lấy từ input: {hits[0]}')
    else:
        raise SystemExit(
            'Thiếu vimqa_200_7b.json.\n'
            'Add Input > Your Work > output của kernel đã sinh nó, HOẶC tải lên\n'
            'thành Dataset rồi gắn vào kernel này.\n'
            'Kernel này KHÔNG tự chạy lại bảng chính: nó cần GPU cho reader 7B,\n'
            'và một kernel CPU sẽ chạy hàng giờ rồi bị hủy.')
print(f'✓ {OUT}')

## 2. Nối hư hại ngữ pháp với F1

Chạy CPU - LLMLingua-2 là BERT-base. Ghép theo chỉ số dòng và **kiểm tra câu hỏi khớp** trước khi tính, không tin thứ tự một cách mù quáng.

In [ ]:
# ══ 1c′ - TƯƠNG QUAN Ở READER 7B ══
import subprocess, sys
print(subprocess.run(
    [sys.executable, '-u', 'scripts/damage_vs_f1.py',
     '--eval-json', 'results/vimqa_200_7b.json',
     '--out', 'results/damage_vs_f1_vimqa200_7b.json'],
    capture_output=True, text=True).stdout)

## 3. So với kết quả reader 0.5B

Câu hỏi duy nhất: **hiệu ứng sàn có phải nguyên nhân của kết quả âm không?**

Nếu ρ ở 7B vẫn ~0 trong khi tỉ lệ F1=0 giảm mạnh, thì sàn KHÔNG phải lời giải
thích, và kết quả âm là thật.

In [ ]:
import json

new = json.load(open('results/damage_vs_f1_vimqa200_7b.json'))
ev  = json.load(open('results/vimqa_200_7b.json'))

# Kết quả CPU reader 0.5B, chép từ results/damage_vs_f1_vimqa200.json để so.
OLD = {'itercomp':   {'rho': +0.030, 'lo': -0.11, 'hi': +0.18, 'zero': 59},
       'llmlingua2': {'rho': +0.028, 'lo': -0.10, 'hi': +0.16, 'zero': 66}}

def fmt(rho, lo, hi):
    return f'rho {rho:+.3f} [{lo:+.2f}, {hi:+.2f}]'

print(f"{'phương pháp':12s} {'reader 0.5B (CPU)':>26s} {'reader 7B':>26s}")
for m, o in OLD.items():
    d = new['summary'][m]
    f1s = [r['methods'][m]['f1_norm'] for r in ev['per_row']]
    zero = 100 * sum(1 for x in f1s if x == 0) / len(f1s)
    print(f"{m:12s} {fmt(o['rho'], o['lo'], o['hi']):>26s}"
          f" {fmt(d['rho'], d['ci_lo'], d['ci_hi']):>26s}")
    old_zero = f"F1=0: {o['zero']}%"
    new_zero = f"F1=0: {zero:.0f}%"
    print(f"{'':12s} {old_zero:>26s} {new_zero:>26s}")

sig = [m for m in OLD
       if new['summary'][m]['ci_hi'] < 0 or new['summary'][m]['ci_lo'] > 0]
print()
if sig:
    print(f"CÓ tương quan ở 7B ({', '.join(sig)}).")
    print("Hiệu ứng sàn ĐÚNG là nguyên nhân của kết quả âm cũ - mục 2.3 của")
    print("proposal viết được lập luận nhân quả.")
else:
    print("VẪN không tương quan ở 7B.")
    print("Sàn KHÔNG phải lời giải thích, nên kết quả âm là thật: hư hại ngữ")
    print("pháp không giải thích được chênh lệch F1. Mục 2.3 giữ cách viết dè dặt.")

## 4. Tải kết quả về

File trong `/kaggle/working` tự xuất hiện ở tab **Output** khi commit notebook.

In [ ]:
import shutil, os
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
z = shutil.make_archive(os.path.join(BASE, 'results_1cprime'), 'zip', 'results')
print(f'✓ {z}  ({os.path.getsize(z)/1e6:.1f} MB)')
for f in sorted(os.listdir('results')):
    print('  ', f)